# Student Score Prediction - Feedforward Neural Network

Predict student final scores based on 8 input marks using PyTorch FNN.

Dataset: Student Performance (Kaggle)

## Setup

In [7]:
!pip install kaggle --quiet

import os, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {DEVICE}')

Using device: cpu


## Kaggle Authentication

Upload your kaggle.json to Colab's file browser first, then run this cell.

In [10]:
# Find and use uploaded kaggle.json
import glob
kaggle_files = glob.glob('/content/kaggle*.json')

if kaggle_files:
    kaggle_file = kaggle_files[0]
    print(f'Using: {kaggle_file}')
    
    os.makedirs('/root/.kaggle', exist_ok=True)
    with open(kaggle_file, 'r') as src:
        with open('/root/.kaggle/kaggle.json', 'w') as dst:
            dst.write(src.read())
    os.chmod('/root/.kaggle/kaggle.json', 0o600)
    print('Kaggle API configured')
else:
    print('ERROR: kaggle.json not found in /content/')
    print('Please upload kaggle.json to the file browser and re-run this cell')

ERROR: kaggle.json not found in /content/
Please upload kaggle.json to the file browser and re-run this cell


In [ ]:
# Solution
def solve(problem):
    """
    Solve the given problem.
    
    Args:
        problem: The problem to solve
        
    Returns:
        The solution to the problem
    """
    # Implement solution here
    pass

# Example usage
result = solve(None)
print(result)


## Download Dataset

In [ ]:
!kaggle datasets download -d larsen0966/student-performance-data-set -p /content/data --unzip -q
print('Dataset downloaded')
print('\nFiles in /content/data:')
!ls -la /content/data

## Load Data

In [ ]:
import glob
csv_files = glob.glob('/content/data/**/*.csv', recursive=True)
print(f'Found {len(csv_files)} CSV files:')
for f in csv_files:
    print(f'  {f}')

if not csv_files:
    print('\nERROR: No CSV files found!')
    !ls -R /content/data
else:
    # Prefer student-mat.csv, fallback to student-por.csv, then any CSV
    mat_files = [f for f in csv_files if 'student-mat.csv' in f]
    por_files = [f for f in csv_files if 'student-por.csv' in f]
    
    if mat_files:
        mat_file = mat_files[0]
        print(f'\nUsing Math dataset: {mat_file}')
    elif por_files:
        mat_file = por_files[0]
        print(f'\nUsing Portuguese dataset: {mat_file}')
    else:
        mat_file = csv_files[0]
        print(f'\nUsing: {mat_file}')
    
    # Read with semicolon separator
    df = pd.read_csv(mat_file, sep=';')
    print(f'\nDataset shape: {df.shape}')
    print(f'Columns: {list(df.columns[:10])}...')  # Show first 10 columns
    df.head()

## Prepare Data - 8 Input Features

We'll use 8 numeric features as input marks to predict G3 (final grade).

In [ ]:
# Select 8 input features (marks/scores)
INPUT_FEATURES = ['G1', 'G2', 'age', 'Medu', 'Fedu', 'studytime', 'failures', 'absences']
TARGET = 'G3'

X = df[INPUT_FEATURES].values.astype(np.float32)
y = df[TARGET].values.astype(np.float32)

print(f'Input shape: {X.shape}')
print(f'Target shape: {y.shape}')
print(f'\nInput features: {INPUT_FEATURES}')

## Train/Val/Test Split

In [ ]:
# Split: 70% train, 15% val, 15% test
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, random_state=42)

# Standardize
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

print(f'Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}')

## Convert to PyTorch Tensors

In [ ]:
def to_tensor(X, y):
    return (torch.tensor(X, dtype=torch.float32).to(DEVICE),
            torch.tensor(y, dtype=torch.float32).unsqueeze(1).to(DEVICE))

X_tr, y_tr = to_tensor(X_train, y_train)
X_vl, y_vl = to_tensor(X_val, y_val)
X_te, y_te = to_tensor(X_test, y_test)

train_loader = DataLoader(TensorDataset(X_tr, y_tr), batch_size=32, shuffle=True)
val_loader = DataLoader(TensorDataset(X_vl, y_vl), batch_size=32)

## Build Feedforward Neural Network

In [ ]:
class StudentFNN(nn.Module):
    def __init__(self, input_dim=8):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.3),
            
            nn.Linear(64, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Dropout(0.2),
            
            nn.Linear(32, 16),
            nn.ReLU(),
            
            nn.Linear(16, 1)
        )
    
    def forward(self, x):
        return self.net(x)

model = StudentFNN(input_dim=8).to(DEVICE)
print(model)
print(f'\nParameters: {sum(p.numel() for p in model.parameters()):,}')

## Train Model

In [ ]:
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=10)

EPOCHS = 200
PATIENCE = 20
best_val_loss = float('inf')
patience_ctr = 0
history = {'train_loss': [], 'val_loss': []}

for epoch in range(1, EPOCHS + 1):
    # Train
    model.train()
    train_loss = 0.0
    for Xb, yb in train_loader:
        optimizer.zero_grad()
        pred = model(Xb)
        loss = criterion(pred, yb)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * Xb.size(0)
    train_loss /= len(train_loader.dataset)
    
    # Validate
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for Xb, yb in val_loader:
            pred = model(Xb)
            val_loss += criterion(pred, yb).item() * Xb.size(0)
    val_loss /= len(val_loader.dataset)
    
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    scheduler.step(val_loss)
    
    if epoch % 20 == 0:
        print(f'Epoch {epoch}/{EPOCHS} | Train: {train_loss:.4f} | Val: {val_loss:.4f}')
    
    # Early stopping
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_weights = {k: v.clone() for k, v in model.state_dict().items()}
        patience_ctr = 0
    else:
        patience_ctr += 1
        if patience_ctr >= PATIENCE:
            print(f'Early stopping at epoch {epoch}')
            break

model.load_state_dict(best_weights)
print('Training complete')

## Training Curves

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(history['train_loss'], label='Train Loss')
plt.plot(history['val_loss'], label='Val Loss')
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.title('Training History')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## Evaluate on Test Set

In [ ]:
model.eval()
with torch.no_grad():
    y_pred = model(X_te).cpu().numpy().flatten()

y_true = y_test

mae = mean_absolute_error(y_true, y_pred)
rmse = np.sqrt(mean_squared_error(y_true, y_pred))
r2 = r2_score(y_true, y_pred)

print('Test Results:')
print(f'MAE:  {mae:.3f}')
print(f'RMSE: {rmse:.3f}')
print(f'R²:   {r2:.4f}')

## Visualize Predictions

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Actual vs Predicted
axes[0].scatter(y_true, y_pred, alpha=0.6, edgecolors='white')
lim = [min(y_true.min(), y_pred.min()) - 1, max(y_true.max(), y_pred.max()) + 1]
axes[0].plot(lim, lim, 'r--', lw=2, label='Perfect prediction')
axes[0].set_xlabel('Actual Score')
axes[0].set_ylabel('Predicted Score')
axes[0].set_title(f'Actual vs Predicted (R²={r2:.3f})')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Residuals
residuals = y_true - y_pred
axes[1].hist(residuals, bins=20, edgecolor='white', alpha=0.8)
axes[1].axvline(0, color='red', linestyle='--')
axes[1].set_xlabel('Residual (Actual - Predicted)')
axes[1].set_ylabel('Count')
axes[1].set_title('Residual Distribution')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## Predict New Student Score

Input 8 marks to predict final score.

In [ ]:
# Edit these 8 input values
new_student = {
    'G1': 12,          # Period 1 grade
    'G2': 13,          # Period 2 grade
    'age': 17,         # Age
    'Medu': 3,         # Mother education (0-4)
    'Fedu': 2,         # Father education (0-4)
    'studytime': 3,    # Study time (1-4)
    'failures': 0,     # Past failures (0-3)
    'absences': 4      # Number of absences
}

# Prepare and predict
student_df = pd.DataFrame([new_student])[INPUT_FEATURES]
student_arr = scaler.transform(student_df.values.astype(np.float32))
student_t = torch.tensor(student_arr, dtype=torch.float32).to(DEVICE)

model.eval()
with torch.no_grad():
    predicted_score = model(student_t).item()

predicted_score = max(0, min(20, predicted_score))

print('Student Score Prediction')
print('=' * 40)
for k, v in new_student.items():
    print(f'{k:12s}: {v}')
print('=' * 40)
print(f'Predicted G3: {predicted_score:.2f} / 20')
print(f'Status: {"Pass" if predicted_score >= 10 else "At Risk"}')

## Save Model

In [ ]:
import pickle

torch.save(model.state_dict(), 'student_fnn.pth')
with open('scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

print('Model saved: student_fnn.pth')
print('Scaler saved: scaler.pkl')